# CISB5123 Text Analytics - Lab 9: Topic Modeling

## EXAMPLE 1 – Basic LDA with Short Documents

In [ ]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel
import pandas as pd

# Download NLTK Resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
documents = [
    "Rafael Nadal Joins Roger Federer in Missing U.S. Open",
    "Rafael Nadal Is Out of the Australian Open",
    "Biden Announces Virus Measures",
    "Biden's Virus Plans Meet Reality",
    "Where Biden's Virus Plan Stands"
]

In [ ]:
stop_words = set(stopwords.words('english'))  # Create a set of English stopwords
lemmatizer = WordNetLemmatizer()              # Initialize a WordNet Lemmatizer

def preprocess_text(text):
    tokens = word_tokenize(text.lower())                                    # Tokenize and lowercase
    tokens = [token for token in tokens if token.isalnum()]                # Remove non-alphanumeric
    tokens = [token for token in tokens if token not in stop_words]        # Remove stopwords
    tokens = [lemmatizer.lemmatize(token) for token in tokens]             # Lemmatize
    return tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]
preprocessed_documents

[['rafael', 'nadal', 'join', 'roger', 'federer', 'missing', 'open'],
 ['rafael', 'nadal', 'australian', 'open'],
 ['biden', 'announces', 'virus', 'measure'],
 ['biden', 'virus', 'plan', 'meet', 'reality'],
 ['biden', 'virus', 'plan', 'stand']]

In [ ]:
# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(preprocessed_documents)

# Convert each preprocessed document into a bag-of-words representation using the dictionary
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

In [ ]:
# corpus: bag-of-words representation of the documents
# num_topics: number of topics to be extracted by the model
# id2word=dictionary: dictionary mapping from word IDs to words
# passes: number of passes through the corpus during training
# Train an LDA model on the corpus with 2 topics using Gensim's LdaModel class
lda_model = LdaModel(corpus, num_topics=2, id2word=dictionary, passes=15)

In [ ]:
# empty list to store dominant topic labels for each document
article_labels = []

# iterate over each processed document
for i, doc in enumerate(preprocessed_documents):
    # for each document, convert to bag-of-words representation
    bow = dictionary.doc2bow(doc)
    # get list of topic probabilities
    topics = lda_model.get_document_topics(bow)
    # determine topic with highest probability
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    # append to the list
    article_labels.append(dominant_topic)

# Create DataFrame
df = pd.DataFrame({"Article": documents, "Topic": article_labels})

# Print the DataFrame
print("Table with Articles and Topic:")
print(df)
print()

Table with Articles and Topic:
                                             Article  Topic
0  Rafael Nadal Joins Roger Federer in Missing U....      1
1         Rafael Nadal Is Out of the Australian Open      1
2                      Biden Announces Virus Measures      0
3                    Biden's Virus Plans Meet Reality      0
4                     Where Biden's Virus Plan Stands      0


In [ ]:
# Print the top terms for each topic
print("Top Terms for Each Topic:")
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

Top Terms for Each Topic:
Topic 0:
- "biden" (weight: 0.166)
- "virus" (weight: 0.166)
- "plan" (weight: 0.118)
- "meet" (weight: 0.071)
- "reality" (weight: 0.071)
- "stand" (weight: 0.071)
- "measure" (weight: 0.071)
- "announces" (weight: 0.071)
- "roger" (weight: 0.025)
- "federer" (weight: 0.025)

Topic 1:
- "nadal" (weight: 0.131)
- "rafael" (weight: 0.131)
- "open" (weight: 0.131)
- "australian" (weight: 0.079)
- "join" (weight: 0.078)
- "missing" (weight: 0.078)
- "federer" (weight: 0.078)
- "roger" (weight: 0.078)
- "announces" (weight: 0.027)
- "measure" (weight: 0.027)


---
## EXAMPLE 2 – LDA with NPR Articles (npr.csv)

In [ ]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel
import pandas as pd

# Download NLTK Resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
df = pd.read_csv('npr.csv')
documents = df['Article'].tolist()

In [ ]:
stop_words = set(stopwords.words('english'))  # Create a set of English stopwords
lemmatizer = WordNetLemmatizer()              # Initialize a WordNet Lemmatizer

def preprocess_text(text):
    tokens = word_tokenize(text.lower())                                    # Tokenize and lowercase
    tokens = [token for token in tokens if token.isalnum()]                # Remove non-alphanumeric
    tokens = [token for token in tokens if token not in stop_words]        # Remove stopwords
    tokens = [lemmatizer.lemmatize(token) for token in tokens]             # Lemmatize
    return tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]
print(preprocessed_documents[0])

['washington', 'policy', 'bipartisan', 'politics', 'even', 'cannot', 'make', 'sense', 'show', 'little', 'sign', 'improving', 'year', 'president', 'trump', 'come', 'office', 'pledging', 'bring', 'people', 'together', 'washington', 'still', 'bitterly', 'divided', 'partisan', 'line', 'vast', 'majority', 'republican', 'still', 'support', 'trump', 'administration', 'approval', 'rating', 'democrat', 'abysmal', 'low', 'health', 'care', 'legislation', 'uniting', 'republican', 'even', 'passing', 'tax', 'reform', 'would', 'likely', 'party', 'line', 'vote']


In [ ]:
# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(preprocessed_documents)

# Filter out tokens that appear in less than 15 documents or more than 50% of the documents
dictionary.filter_extremes(no_below=15, no_above=0.5)

# Convert each preprocessed document into a bag-of-words representation using the dictionary
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

In [ ]:
# Run LDA
lda_model = LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15)

In [ ]:
# empty list to store dominant topic labels for each document
article_labels = []

# iterate over each processed document
for i, doc in enumerate(preprocessed_documents):
    # for each document, convert to bag-of-words representation
    bow = dictionary.doc2bow(doc)
    # get list of topic probabilities
    topics = lda_model.get_document_topics(bow)
    # determine topic with highest probability
    dominant_topic = max(topics, key=lambda x: x[1])[0] if topics else -1
    # append to the list
    article_labels.append(dominant_topic)

# Create DataFrame
df_result = pd.DataFrame({"Article": documents, "Topic": article_labels})

# Print the DataFrame
print("Table with Articles and Topic:")
print(df_result)
print()

Table with Articles and Topic:
                                                 Article  Topic
0      In the Washington of 2016, even when the polic...      2
1        Donald Trump has used Twitter  —   his prefe...      2
2        Donald Trump is unabashedly praising Russian...      2
3      Updated at 2:50 p. m. ET, Russian President Vl...      2
4      From photography, illustration and video, to d...      0
...                                                  ...    ...
11987  The number of law enforcement officers shot an...      4
11988    Trump is busy these days with victory tours,...      2
11989  It's always interesting for the Goats and Soda...      0
11990  The election of Donald Trump was a surprise to...      2
11991  Voters in the English city of Sunderland did s...      4

[11992 rows x 2 columns]


In [ ]:
# Print top terms for each topic
for topic_id in range(lda_model.num_topics):
    print(f"Top terms for Topic #{topic_id}:")
    top_terms = lda_model.show_topic(topic_id, topn=10)
    print([term[0] for term in top_terms])
    print()

Top terms for Topic #0:
['food', 'school', 'student', 'company', 'water', 'percent', 'world', 'job', 'university', 'much']

Top terms for Topic #1:
['know', 'think', 'thing', 'life', 'really', 'woman', 'story', 'show', 'book', 'something']

Top terms for Topic #2:
['police', 'country', 'report', 'city', 'government', 'attack', 'told', 'war', 'two', 'state']

Top terms for Topic #3:
['health', 'state', 'care', 'law', 'case', 'study', 'child', 'drug', 'patient', 'percent']

Top terms for Topic #4:
['trump', 'clinton', 'president', 'state', 'republican', 'campaign', 'election', 'obama', 'vote', 'house']


In [ ]:
# Print the top terms for each topic with weight
print("Top Terms for Each Topic:")
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        parts = term.split("*")
        if len(parts) == 2:
            weight, word = parts
            print(f"- {word.strip()} (weight: {weight.strip()})")
    print()

Top Terms for Each Topic:
Topic 0:
- "food" (weight: 0.006)
- "school" (weight: 0.006)
- "student" (weight: 0.005)
- "company" (weight: 0.005)
- "water" (weight: 0.004)
- "percent" (weight: 0.004)
- "world" (weight: 0.003)
- "job" (weight: 0.003)
- "university" (weight: 0.003)
- "much" (weight: 0.003)

Topic 1:
- "know" (weight: 0.005)
- "think" (weight: 0.005)
- "thing" (weight: 0.005)
- "life" (weight: 0.005)
- "really" (weight: 0.004)
- "woman" (weight: 0.004)
- "story" (weight: 0.004)
- "show" (weight: 0.003)
- "book" (weight: 0.003)
- "something" (weight: 0.003)

Topic 2:
- "police" (weight: 0.007)
- "country" (weight: 0.006)
- "report" (weight: 0.005)
- "city" (weight: 0.005)
- "government" (weight: 0.004)
- "attack" (weight: 0.004)
- "told" (weight: 0.004)
- "war" (weight: 0.004)
- "two" (weight: 0.004)
- "state" (weight: 0.004)

Topic 3:
- "health" (weight: 0.011)
- "state" (weight: 0.007)
- "care" (weight: 0.006)
- "law" (weight: 0.006)
- "case" (weight: 0.006)
- "study" (weig